# Dataset 3 — Embedding threshold sensitivity (node2vec_v2_32 + MLP tuned)

Uses the **per-p** `node2vec_v2_32` embeddings (re-extracted for each threshold; see
`03_embeddings/dataset_3/03_embeddings_p_thresholds.ipynb`). For each p the selected MLP is
re-fit on that p's embeddings + target with the same 70/15/15 stratified split.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / 'src').exists() and (p / 'requirements.txt').exists():
            return p
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from src.models.ml_train_and_store import load_model, random_split, CLASSICAL_FEATURE_CANDIDATES

DATASET = 'dataset_3'
APPROACH = 'embedding'
TARGET_COL = 'log_systemic_risk_label'
P_LIST = ['p0', 'p5', 'p10', 'p15', 'p20', 'p25', 'p30', 'p35', 'p40']

TARGETS_DIR = PROJECT_ROOT / 'src' / 'datasets' / 'dataset_3' / 'targets'
SRC = PROJECT_ROOT / 'src' / 'data' / 'embeddings' / 'dataset_3' / 'network_based' / 'node2vec_v2_32_dataset3_dataset.parquet'
MODEL_PATH  = PROJECT_ROOT / 'src' / 'models' / 'dataset_3/05_a/node2vec_v2_32/MLP_(tuned).joblib'
OUT_DIR     = PROJECT_ROOT / 'src' / 'data' / 'predictions' / 'dataset_3/embedding_threshold'
OUT_DIR.mkdir(parents=True, exist_ok=True)

model = load_model(MODEL_PATH)
print('model:', MODEL_PATH.name, '| approach:', APPROACH)

model: MLP_(tuned).joblib | approach: embedding


## Apply the selected model across thresholds

In [2]:
def load_p(p):
    # Per-p embeddings (node2vec_v2_32 re-extracted for each p; target already merged in).
    # p0 uses the baseline file; p5..p40 use the per-threshold files from
    # 03_embeddings/dataset_3/03_embeddings_p_thresholds.ipynb.
    name = SRC if p == 'p0' else SRC.with_name(f'node2vec_v2_32_dataset3_{p}_dataset.parquet')
    df = pd.read_parquet(name)
    ecols = [c for c in df.columns if c.startswith('emb_')]
    return df[['bank_id'] + ecols + [TARGET_COL]].dropna(subset=[TARGET_COL]).reset_index(drop=True), ecols

In [3]:
rows, preds = [], []
for p in P_LIST:
    df, fcols = load_p(p)
    tr, va, te = random_split(df, TARGET_COL)            # 70/15/15 stratified
    m = clone(model).fit(tr[fcols], tr[TARGET_COL])      # same selected model, refit at this threshold
    row = {'p': p}
    for split_name, sdf in [('train', tr), ('validation', va), ('test', te)]:
        yp = m.predict(sdf[fcols])
        row[f'{split_name}_rmse'] = mean_squared_error(sdf[TARGET_COL], yp) ** 0.5
        row[f'{split_name}_mae']  = mean_absolute_error(sdf[TARGET_COL], yp)
        pf = sdf[['bank_id', TARGET_COL]].copy()
        pf['prediction'] = yp; pf['split'] = split_name; pf['p'] = p
        pf['dataset'] = DATASET; pf['approach'] = APPROACH
        preds.append(pf)
    row['val/train_rmse'] = round(row['validation_rmse'] / row['train_rmse'], 2)
    row['val/train_mae']  = round(row['validation_mae'] / row['train_mae'], 2)
    rows.append(row)

metrics = pd.DataFrame(rows)
predictions = pd.concat(preds, ignore_index=True)
metrics.to_csv(OUT_DIR / 'metrics.csv', index=False)
predictions.to_csv(OUT_DIR / 'predictions.csv', index=False)
print('saved ->', OUT_DIR)
display(metrics.round(3))

saved -> /Users/rubenmarques/Documents/Repositórios/Thesis/src/data/predictions/dataset_3/embedding_threshold


,p,train_rmse,train_mae,validation_rmse,validation_mae,test_rmse,test_mae,val/train_rmse,val/train_mae
0,p0,0.577,0.380,0.682,0.451,0.694,0.455,1.18,1.19
1,p5,0.543,0.354,0.712,0.462,0.692,0.457,1.31,1.30
2,p10,0.582,0.386,0.702,0.463,0.702,0.466,1.21,1.20
3,p15,0.606,0.394,0.698,0.458,0.700,0.459,1.15,1.16
4,p20,0.550,0.356,0.749,0.489,0.734,0.467,1.36,1.37
5,p25,0.553,0.361,0.740,0.491,0.752,0.483,1.34,1.36
6,p30,0.522,0.333,0.768,0.499,0.763,0.483,1.47,1.50
7,p35,0.538,0.350,0.750,0.496,0.745,0.475,1.39,1.42
8,p40,0.543,0.348,0.759,0.488,0.734,0.461,1.40,1.40
